# Audit Active-Satellite Exposure

Propagates a cached active-satellite catalog across the campaign and measures exposure by altitude shell. It also audits the effective pointing action space.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


## What is real and what is simulated


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, io, json, math, os, subprocess, sys, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


ACTION_DIR = PROJECT_ROOT / 'phase1_outputs' / 'ordem_action_library' / '33aa70ca_80_plus_baseline'
ACTION_RESPONSE_PATH = ACTION_DIR / 'ordem_action_response_matrix.csv.gz'

OUTPUT_DIR = PROJECT_ROOT / 'phase2_outputs' / 'dynamic_asset_exposure' / 'active_omm_action_audit_v1'
SNAPSHOT_DIR = OUTPUT_DIR / 'snapshots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

                                                                 
ACTIVE_OMM_URL = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=active&FORMAT=CSV'
REFRESH_ACTIVE_SATELLITES = False                                                           
PROPAGATION_STEP_MINUTES = 15
PROPAGATION_HOURS = 24
ALTITUDE_EDGES_KM = np.arange(915.0, 1065.0 + 15.0, 15.0)
INCLINATION_EDGES_DEG = np.arange(0.0, 180.0 + 10.0, 10.0)
PRIMARY_ORDEM_FEATURE = 'window_contribution_gt_1mm_m2_yr'
MIN_ACTIVE_ROWS = int(os.environ.get('DEBRIS_MIN_ACTIVE_ROWS', '1000'))
SKIP_SGP4_INSTALL = os.environ.get('DEBRIS_SKIP_SGP4_INSTALL', '0') == '1'

print('research root:', PROJECT_ROOT)
print('Output folder:', OUTPUT_DIR)
print('Satellite snapshot refresh:', REFRESH_ACTIVE_SATELLITES)
print('Propagation:', PROPAGATION_HOURS, 'hours at', PROPAGATION_STEP_MINUTES, 'minute spacing')
print('CPU runtime is correct; a GPU is unnecessary.')


## 1. Audit the 81 ORDEM choices


In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


if not ACTION_RESPONSE_PATH.is_file():
    raise FileNotFoundError(f'Missing completed ORDEM response matrix: {ACTION_RESPONSE_PATH}')

response = pd.read_csv(ACTION_RESPONSE_PATH)
required_response_columns = {
    'action_id', 'azimuth_deg', 'elevation_deg', 'alt_low', 'alt_high',
    'range_overlap_fraction', PRIMARY_ORDEM_FEATURE,
}
missing = required_response_columns - set(response.columns)
if missing:
    raise KeyError(f'ORDEM response matrix is missing columns: {sorted(missing)}')
if response.action_id.nunique() != 81:
    raise RuntimeError(f'Expected 81 actions; found {response.action_id.nunique()}')

active = response.loc[response.range_overlap_fraction > 0].copy()
active['altitude_mid_km'] = 0.5 * (active.alt_low + active.alt_high)
active['shell_index'] = np.searchsorted(ALTITUDE_EDGES_KM, active.altitude_mid_km, side='right') - 1
active = active.loc[active.shell_index.between(0, len(ALTITUDE_EDGES_KM) - 2)].copy()

action_meta = (response[['action_id', 'azimuth_deg', 'elevation_deg']]
               .drop_duplicates().sort_values(['elevation_deg', 'azimuth_deg']).reset_index(drop=True))
action_ids = action_meta.action_id.tolist()
action_index = {name: idx for idx, name in enumerate(action_ids)}
n_shells = len(ALTITUDE_EDGES_KM) - 1
H = np.zeros((len(action_ids), n_shells), dtype=float)
for (action_id, shell_index), group in active.groupby(['action_id', 'shell_index']):
    H[action_index[action_id], int(shell_index)] = group[PRIMARY_ORDEM_FEATURE].sum()

if not np.isfinite(H).all() or (H < 0).any() or np.any(H.sum(axis=1) <= 0):
    raise RuntimeError('The reconstructed action-response matrix contains invalid rows.')

row_norm = H / np.linalg.norm(H, axis=1, keepdims=True)
similarity = np.clip(row_norm @ row_norm.T, -1.0, 1.0)

def effective_rank(matrix):
    singular = np.linalg.svd(matrix, compute_uv=False)
    energy = singular ** 2
    probability = energy / energy.sum()
    return float(np.exp(-(probability * np.log(np.maximum(probability, 1e-300))).sum())), singular


raw_effective_rank, raw_singular = effective_rank(H)
shape_effective_rank, shape_singular = effective_rank(row_norm)
shape_energy = shape_singular ** 2
components_99 = int(np.searchsorted(np.cumsum(shape_energy) / shape_energy.sum(), 0.99) + 1)

same_elevation_pairs = []
for elevation, indices in action_meta.groupby('elevation_deg').groups.items():
    indices = np.asarray(list(indices), dtype=int)
    values = similarity[np.ix_(indices, indices)][np.triu_indices(len(indices), k=1)]
    same_elevation_pairs.extend(values.tolist())

same_azimuth_pairs = []
for azimuth, indices in action_meta.groupby('azimuth_deg').groups.items():
    indices = np.asarray(list(indices), dtype=int)
    values = similarity[np.ix_(indices, indices)][np.triu_indices(len(indices), k=1)]
    same_azimuth_pairs.extend(values.tolist())

elevation_rows = []
effective_H = []
for elevation, group in action_meta.groupby('elevation_deg', sort=True):
    indices = group.index.to_numpy(dtype=int)
    within = similarity[np.ix_(indices, indices)][np.triu_indices(len(indices), k=1)]
    mean_response = H[indices].mean(axis=0)
    effective_H.append(mean_response)
    representative = group.iloc[np.abs(group.azimuth_deg.to_numpy() - 90.0).argmin()]
    elevation_rows.append({
        'effective_action_id': f'elevation_{elevation:g}_deg',
        'elevation_deg': float(elevation),
        'source_actions': int(len(indices)),
        'representative_action_id': representative.action_id,
        'mean_total_ordem_response': float(mean_response.sum()),
        'minimum_same_elevation_azimuth_similarity': float(within.min()),
        'median_same_elevation_azimuth_similarity': float(np.median(within)),
    })

effective_H = np.asarray(effective_H)
effective_actions = pd.DataFrame(elevation_rows)
effective_long = []
for idx, row in effective_actions.iterrows():
    for shell_index in range(n_shells):
        effective_long.append({
            'effective_action_id': row.effective_action_id,
            'elevation_deg': row.elevation_deg,
            'shell_index': shell_index,
            'altitude_low_km': ALTITUDE_EDGES_KM[shell_index],
            'altitude_high_km': ALTITUDE_EDGES_KM[shell_index + 1],
            'ordem_response_gt_1mm_m2_yr': effective_H[idx, shell_index],
        })
effective_long = pd.DataFrame(effective_long)

action_audit = {
    'raw_actions': 81,
    'effective_elevation_families': int(len(effective_actions)),
    'raw_effective_rank': raw_effective_rank,
    'row_normalized_effective_rank': shape_effective_rank,
    'components_explaining_99_percent_shape_energy': components_99,
    'same_elevation_different_azimuth_similarity_min': float(np.min(same_elevation_pairs)),
    'same_elevation_different_azimuth_similarity_median': float(np.median(same_elevation_pairs)),
    'same_azimuth_different_elevation_similarity_min': float(np.min(same_azimuth_pairs)),
    'same_azimuth_different_elevation_similarity_median': float(np.median(same_azimuth_pairs)),
    'supports_81_independent_action_claim': False,
    'supports_elevation_family_adaptation': True,
    'interpretation': 'Azimuth changes inside the existing 80-100 degree grid barely change the ORDEM altitude response; elevation is the meaningful action dimension.',
}

effective_actions.to_csv(OUTPUT_DIR / 'ordem_effective_elevation_actions.csv', index=False)
effective_long.to_csv(OUTPUT_DIR / 'ordem_effective_elevation_response_matrix.csv', index=False)
(OUTPUT_DIR / 'ordem_action_identifiability_audit.json').write_text(json.dumps(action_audit, indent=2))

print('ACTION AUDIT COMPLETE')
print('  Raw labels:', action_audit['raw_actions'])
print('  Effective elevation families:', action_audit['effective_elevation_families'])
print('  Normalized effective rank:', f"{shape_effective_rank:.2f}")
print('  Components explaining 99%:', components_99)
print('  Same-elevation azimuth similarity:', f"{np.median(same_elevation_pairs):.6f}", '(median)')
display(effective_actions)


### Action-audit interpretation


## 2. Download once, then cache the real active-satellite catalog


In [ ]:
from urllib.request import Request, urlopen

required_omm_columns = {
    'OBJECT_NAME', 'OBJECT_ID', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY',
    'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY',
    'NORAD_CAT_ID',
}

cached_snapshots = sorted(SNAPSHOT_DIR.glob('active_omm_*.csv'))
downloaded_now = False

if cached_snapshots and not REFRESH_ACTIVE_SATELLITES:
    snapshot_path = cached_snapshots[-1]
    print('Using cached active-satellite snapshot:', snapshot_path.name)
else:
    request_time = datetime.now(timezone.utc)
    print('Downloading one current active-satellite OMM CSV from CelesTrak...')
    request = Request(ACTIVE_OMM_URL, headers={'User-Agent': 'research-research-active-orbit-cache/1.0'})
    with urlopen(request, timeout=120) as reply:
        content = reply.read()
    if len(content) < 10_000:
        raise RuntimeError('CelesTrak response is unexpectedly small; refusing to cache it.')
    snapshot_path = SNAPSHOT_DIR / f"active_omm_{request_time.strftime('%Y%m%dT%H%M%SZ')}.csv"
    snapshot_path.write_bytes(content)
    downloaded_now = True
    metadata = {
        'source_url': ACTIVE_OMM_URL,
        'retrieved_utc': request_time.isoformat(),
        'sha256': sha256_file(snapshot_path),
        'bytes': snapshot_path.stat().st_size,
        'request_policy': 'one request; future notebook reruns reuse the repository cache unless REFRESH_ACTIVE_SATELLITES=True',
    }
    snapshot_path.with_suffix('.metadata.json').write_text(json.dumps(metadata, indent=2))

active_omm = pd.read_csv(snapshot_path, dtype={'NORAD_CAT_ID': str, 'OBJECT_ID': str})
missing = required_omm_columns - set(active_omm.columns)
if missing:
    raise KeyError(f'Active OMM snapshot is missing fields: {sorted(missing)}')
if len(active_omm) < MIN_ACTIVE_ROWS:
    raise RuntimeError(f'Only {len(active_omm)} active-satellite rows were found; expected at least {MIN_ACTIVE_ROWS:,}.')
if active_omm.NORAD_CAT_ID.duplicated().any():
    active_omm = active_omm.sort_values('EPOCH').drop_duplicates('NORAD_CAT_ID', keep='last').reset_index(drop=True)

metadata_path = snapshot_path.with_suffix('.metadata.json')
if metadata_path.is_file():
    snapshot_metadata = json.loads(metadata_path.read_text())
else:
    snapshot_metadata = {
        'source_url': ACTIVE_OMM_URL,
        'retrieved_utc': datetime.fromtimestamp(snapshot_path.stat().st_mtime, tz=timezone.utc).isoformat(),
        'sha256': sha256_file(snapshot_path),
        'bytes': snapshot_path.stat().st_size,
        'note': 'Metadata sidecar reconstructed from the cached file modification time.',
    }
    metadata_path.write_text(json.dumps(snapshot_metadata, indent=2))

print('ACTIVE CATALOG AUDIT PASSED')
print('  Rows:', f'{len(active_omm):,}')
print('  Snapshot:', snapshot_path.name)
print('  SHA-256:', sha256_file(snapshot_path))
print('  Downloaded during this run:', downloaded_now)
display(active_omm[['OBJECT_NAME', 'NORAD_CAT_ID', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION']].head())


## 3. Propagate the real active satellites for 24 hours


In [ ]:
try:
    from sgp4.api import Satrec, jday
    from sgp4 import omm
    SGP4_AVAILABLE = True
except ImportError:
    print('Installing the small sgp4 package...')
    try:
        if SKIP_SGP4_INSTALL:
            raise RuntimeError('SGP4 installation skipped by DEBRIS_SKIP_SGP4_INSTALL=1')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sgp4'])
        from sgp4.api import Satrec, jday
        from sgp4 import omm
        SGP4_AVAILABLE = True
    except Exception as exc:
        warnings.warn(f'SGP4 installation failed; using mean-element fallback for all rows: {exc}')
        SGP4_AVAILABLE = False

retrieved_utc = pd.Timestamp(snapshot_metadata['retrieved_utc'])
if retrieved_utc.tzinfo is None:
    retrieved_utc = retrieved_utc.tz_localize('UTC')
else:
    retrieved_utc = retrieved_utc.tz_convert('UTC')
propagation_start = retrieved_utc.floor(f'{PROPAGATION_STEP_MINUTES}min')
samples_per_hour = 60 // PROPAGATION_STEP_MINUTES
sample_count = PROPAGATION_HOURS * samples_per_hour
sample_times = pd.date_range(
    propagation_start,
    periods=sample_count,
    freq=f'{PROPAGATION_STEP_MINUTES}min',
    tz='UTC',
)

if SGP4_AVAILABLE:
    seconds = sample_times.second.to_numpy(dtype=float) + sample_times.microsecond.to_numpy(dtype=float) / 1e6
    jd, fr = jday(
        sample_times.year.to_numpy(), sample_times.month.to_numpy(), sample_times.day.to_numpy(),
        sample_times.hour.to_numpy(), sample_times.minute.to_numpy(), seconds,
    )

MU_EARTH_KM3_S2 = 398600.4418
EARTH_RADIUS_KM = 6378.137

def mean_element_fallback(fields):
    mean_motion_rev_day = float(fields['MEAN_MOTION'])
    eccentricity = float(fields['ECCENTRICITY'])
    inclination = np.deg2rad(float(fields['INCLINATION']))
    argument_pericenter = np.deg2rad(float(fields['ARG_OF_PERICENTER']))
    mean_anomaly_epoch = np.deg2rad(float(fields['MEAN_ANOMALY']))
    epoch = pd.Timestamp(fields['EPOCH'])
    if epoch.tzinfo is None:
        epoch = epoch.tz_localize('UTC')
    else:
        epoch = epoch.tz_convert('UTC')
    delta_days = (sample_times - epoch).total_seconds().to_numpy() / 86400.0
    mean_anomaly = np.mod(mean_anomaly_epoch + 2 * np.pi * mean_motion_rev_day * delta_days, 2 * np.pi)
    eccentric_anomaly = mean_anomaly.copy()
    for _ in range(8):
        eccentric_anomaly -= (
            eccentric_anomaly - eccentricity * np.sin(eccentric_anomaly) - mean_anomaly
        ) / np.maximum(1 - eccentricity * np.cos(eccentric_anomaly), 1e-10)
    true_anomaly = 2 * np.arctan2(
        np.sqrt(1 + eccentricity) * np.sin(eccentric_anomaly / 2),
        np.sqrt(1 - eccentricity) * np.cos(eccentric_anomaly / 2),
    )
    mean_motion_rad_s = mean_motion_rev_day * 2 * np.pi / 86400.0
    semimajor_axis = (MU_EARTH_KM3_S2 / mean_motion_rad_s ** 2) ** (1 / 3)
    radius = semimajor_axis * (1 - eccentricity * np.cos(eccentric_anomaly))
    altitude = radius - EARTH_RADIUS_KM
    latitude = np.rad2deg(np.arcsin(np.clip(np.sin(inclination) * np.sin(argument_pericenter + true_anomaly), -1, 1)))
    valid = np.isfinite(altitude) & np.isfinite(latitude) & (altitude > 80) & (altitude < 100_000)
    return altitude, latitude, valid


raw_rows = list(csv.DictReader(io.StringIO(snapshot_path.read_text())))
raw_by_id = {str(row['NORAD_CAT_ID']).strip(): row for row in raw_rows}

n_hours = PROPAGATION_HOURS
n_shells = len(ALTITUDE_EDGES_KM) - 1
n_inc_bins = len(INCLINATION_EDGES_DEG) - 1
hour_shell_samples = np.zeros((n_hours, n_shells), dtype=float)
hour_shell_weighted = np.zeros((n_hours, n_shells), dtype=float)
shell_inclination_samples = np.zeros((n_shells, n_inc_bins), dtype=float)
unique_ids_by_shell = [set() for _ in range(n_shells)]
orbit_summaries = []
propagation_failures = []

                                                                                               
critical_list_path = PROJECT_ROOT / 'inputs' / 'critical_satellites.csv'
priority_by_id = {}
priority_mode = 'all_active_satellites_equal_weight'
if critical_list_path.is_file():
    critical = pd.read_csv(critical_list_path, dtype={'NORAD_CAT_ID': str})
    if 'NORAD_CAT_ID' not in critical:
        raise KeyError('critical_satellites.csv must contain NORAD_CAT_ID.')
    if 'priority_weight' not in critical:
        critical['priority_weight'] = 1.0
    if (critical.priority_weight <= 0).any() or not np.isfinite(critical.priority_weight).all():
        raise ValueError('critical_satellites.csv priority_weight values must be finite and positive.')
    priority_by_id = dict(zip(critical.NORAD_CAT_ID.astype(str), critical.priority_weight.astype(float)))
    priority_mode = 'user_supplied_critical_satellite_weights'

for row_number, metadata_row in active_omm.iterrows():
    catalog_id = str(metadata_row.NORAD_CAT_ID).strip()
    fields = raw_by_id.get(catalog_id)
    if fields is None:
        propagation_failures.append({'NORAD_CAT_ID': catalog_id, 'reason': 'raw OMM row not found'})
        continue

    engine = 'mean_element_fallback'
    altitude = latitude = valid = None
    if SGP4_AVAILABLE:
        try:
            satellite = Satrec()
            omm.initialize(satellite, fields)
            errors, positions, _velocities = satellite.sgp4_array(jd, fr)
            radii = np.linalg.norm(positions, axis=1)
            altitude = radii - EARTH_RADIUS_KM
            latitude = np.rad2deg(np.arcsin(np.clip(positions[:, 2] / radii, -1, 1)))
            valid = (errors == 0) & np.isfinite(altitude) & np.isfinite(latitude)
            engine = 'sgp4'
        except Exception:
            altitude = latitude = valid = None

    if altitude is None:
        try:
            altitude, latitude, valid = mean_element_fallback(fields)
        except Exception as exc:
            propagation_failures.append({'NORAD_CAT_ID': catalog_id, 'reason': repr(exc)})
            continue

    if valid.sum() == 0:
        propagation_failures.append({'NORAD_CAT_ID': catalog_id, 'reason': 'no valid propagated samples'})
        continue

    inclination_deg = float(fields['INCLINATION'])
    priority = float(priority_by_id.get(catalog_id, 1.0 if not priority_by_id else 0.0))
    shell_indices = np.searchsorted(ALTITUDE_EDGES_KM, altitude, side='right') - 1
    hour_indices = np.arange(sample_count) // samples_per_hour
    in_window = valid & (shell_indices >= 0) & (shell_indices < n_shells)

    if in_window.any():
        np.add.at(hour_shell_samples, (hour_indices[in_window], shell_indices[in_window]), 1.0)
        if priority > 0:
            np.add.at(hour_shell_weighted, (hour_indices[in_window], shell_indices[in_window]), priority)
        inc_index = int(np.searchsorted(INCLINATION_EDGES_DEG, inclination_deg, side='right') - 1)
        if 0 <= inc_index < n_inc_bins:
            np.add.at(shell_inclination_samples, (shell_indices[in_window], np.full(in_window.sum(), inc_index)), 1.0)
        for shell_index in np.unique(shell_indices[in_window]):
            unique_ids_by_shell[int(shell_index)].add(catalog_id)

    orbit_summaries.append({
        'OBJECT_NAME': fields.get('OBJECT_NAME', ''),
        'OBJECT_ID': fields.get('OBJECT_ID', ''),
        'NORAD_CAT_ID': catalog_id,
        'OMM_EPOCH': fields.get('EPOCH', ''),
        'inclination_deg': inclination_deg,
        'propagation_engine': engine,
        'valid_samples': int(valid.sum()),
        'minimum_altitude_km': float(np.nanmin(altitude[valid])),
        'median_altitude_km': float(np.nanmedian(altitude[valid])),
        'maximum_altitude_km': float(np.nanmax(altitude[valid])),
        'fraction_of_day_in_radar_altitude_window': float(in_window.sum() / valid.sum()),
        'priority_weight': priority,
    })

orbit_summary = pd.DataFrame(orbit_summaries)
failure_table = pd.DataFrame(propagation_failures)
if len(orbit_summary) < 0.9 * len(active_omm):
    raise RuntimeError(f'Only {len(orbit_summary)} of {len(active_omm)} active satellites propagated successfully.')

sgp4_fraction = float((orbit_summary.propagation_engine == 'sgp4').mean())
if SGP4_AVAILABLE and sgp4_fraction < 0.9:
    warnings.warn(f'Only {sgp4_fraction:.1%} used SGP4; inspect propagation_failures.csv before manuscript use.')

orbit_summary.to_csv(OUTPUT_DIR / 'active_satellite_orbit_summary.csv.gz', index=False, compression='gzip')
failure_table.to_csv(OUTPUT_DIR / 'active_satellite_propagation_failures.csv', index=False)

print('PROPAGATION COMPLETE')
print('  Active satellites propagated:', f'{len(orbit_summary):,}', '/', f'{len(active_omm):,}')
print('  SGP4 fraction:', f'{sgp4_fraction:.1%}')
print('  Satellites entering the 915–1065 km shell window:', f"{(orbit_summary.fraction_of_day_in_radar_altitude_window > 0).sum():,}")
print('  Priority mode:', priority_mode)


## 4. Build hourly asset exposure and the action-overlap diagnostic


In [ ]:
shell_rows = []
total_unweighted = hour_shell_samples.sum()
total_weighted = hour_shell_weighted.sum()
ordem_population = effective_H.mean(axis=0)
ordem_population_weight = ordem_population / ordem_population.sum()

for shell_index in range(n_shells):
    shell_rows.append({
        'shell_index': shell_index,
        'altitude_low_km': ALTITUDE_EDGES_KM[shell_index],
        'altitude_high_km': ALTITUDE_EDGES_KM[shell_index + 1],
        'altitude_center_km': 0.5 * (ALTITUDE_EDGES_KM[shell_index] + ALTITUDE_EDGES_KM[shell_index + 1]),
        'active_satellite_time_samples': int(hour_shell_samples[:, shell_index].sum()),
        'weighted_satellite_time_samples': float(hour_shell_weighted[:, shell_index].sum()),
        'unique_active_satellites': int(len(unique_ids_by_shell[shell_index])),
        'active_asset_exposure_weight': float(hour_shell_weighted[:, shell_index].sum() / total_weighted) if total_weighted > 0 else 0.0,
        'ordem_population_weight': float(ordem_population_weight[shell_index]),
    })
shell_summary = pd.DataFrame(shell_rows)

hourly_rows = []
for hour in range(n_hours):
    denominator = hour_shell_weighted[hour].sum()
    for shell_index in range(n_shells):
        hourly_rows.append({
            'hour_index': hour,
            'start_utc': (propagation_start + pd.Timedelta(hours=hour)).isoformat(),
            'shell_index': shell_index,
            'altitude_low_km': ALTITUDE_EDGES_KM[shell_index],
            'altitude_high_km': ALTITUDE_EDGES_KM[shell_index + 1],
            'weighted_satellite_samples': float(hour_shell_weighted[hour, shell_index]),
            'hourly_asset_exposure_weight': float(hour_shell_weighted[hour, shell_index] / denominator) if denominator > 0 else 0.0,
        })
hourly_exposure = pd.DataFrame(hourly_rows)

inclination_rows = []
for shell_index in range(n_shells):
    for inc_index in range(n_inc_bins):
        inclination_rows.append({
            'shell_index': shell_index,
            'altitude_low_km': ALTITUDE_EDGES_KM[shell_index],
            'altitude_high_km': ALTITUDE_EDGES_KM[shell_index + 1],
            'inclination_low_deg': INCLINATION_EDGES_DEG[inc_index],
            'inclination_high_deg': INCLINATION_EDGES_DEG[inc_index + 1],
            'active_satellite_time_samples': float(shell_inclination_samples[shell_index, inc_index]),
        })
inclination_exposure = pd.DataFrame(inclination_rows)

                                                                            
effective_shapes = effective_H / effective_H.sum(axis=1, keepdims=True)
alignment_rows = []
for hour in range(n_hours):
    denominator = hour_shell_weighted[hour].sum()
    asset_weights = hour_shell_weighted[hour] / denominator if denominator > 0 else np.zeros(n_shells)
    overlaps = effective_shapes @ asset_weights
    for action_index, action in effective_actions.iterrows():
        alignment_rows.append({
            'hour_index': hour,
            'start_utc': (propagation_start + pd.Timedelta(hours=hour)).isoformat(),
            'effective_action_id': action.effective_action_id,
            'elevation_deg': action.elevation_deg,
            'asset_population_shape_overlap': float(overlaps[action_index]),
            'is_hourly_highest_overlap': bool(action_index == int(np.argmax(overlaps))) if denominator > 0 else False,
        })
hourly_action_alignment = pd.DataFrame(alignment_rows)

shell_summary.to_csv(OUTPUT_DIR / 'active_satellite_shell_exposure.csv', index=False)
hourly_exposure.to_csv(OUTPUT_DIR / 'active_satellite_hourly_shell_exposure.csv', index=False)
inclination_exposure.to_csv(OUTPUT_DIR / 'active_satellite_altitude_inclination_exposure.csv', index=False)
hourly_action_alignment.to_csv(OUTPUT_DIR / 'hourly_asset_action_overlap_diagnostic.csv', index=False)

                                                                              
template_path = OUTPUT_DIR / 'critical_satellite_input_template.csv'
if not template_path.exists():
    pd.DataFrame(columns=['NORAD_CAT_ID', 'mission_class', 'priority_weight', 'justification']).to_csv(template_path, index=False)

print('ASSET EXPOSURE TABLES COMPLETE')
display(shell_summary)


## 5. Scientific readiness decision


In [ ]:
satellites_in_window = int((orbit_summary.fraction_of_day_in_radar_altitude_window > 0).sum())
has_asset_exposure = bool(total_weighted > 0 and satellites_in_window > 0)

readiness = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'complete',
    'inputs': {
        'ordem_action_response_path': str(ACTION_RESPONSE_PATH),
        'ordem_action_response_sha256': sha256_file(ACTION_RESPONSE_PATH),
        'active_satellite_snapshot': str(snapshot_path),
        'active_satellite_snapshot_sha256': sha256_file(snapshot_path),
        'active_satellite_source_url': ACTIVE_OMM_URL,
        'active_satellite_rows': int(len(active_omm)),
        'active_satellites_propagated': int(len(orbit_summary)),
        'satellites_intersecting_915_1065_km': satellites_in_window,
        'priority_mode': priority_mode,
    },
    'action_identifiability': action_audit,
    'model_decisions': {
        'use_81_actions_as_independent_choices': False,
        'use_9_elevation_family_actions': True,
        'claim_full_azimuth_direction_learning': False,
        'use_real_active_satellites': has_asset_exposure,
        'use_made_up_satellite_names_or_orbits': False,
        'treat_state_cells_as_orbital_population_shells_not_fixed_sky_locations': True,
        'use_altitude_only_primary_model': True,
        'claim_inclination_resolved_debris_map': False,
        'claim_individual_debris_tracking': False,
        'claim_collision_warning_or_avoidance': False,
    },
    'next_policy_specification': {
        'state': 'uncertain debris population multiplier in 10 orbital altitude shells',
        'actions': '9 empirically distinguishable elevation families under equal dwell time',
        'time_update': 'near-identity 24-hour population model; stress-test eccentric-shell mixing and localized anomalies',
        'utility': 'risk-weighted expected reduction in posterior uncertainty, using real active-satellite exposure weights',
        'baselines': ['fixed elevation', 'random feasible elevation', 'highest ORDEM flux', 'expected uncertainty reduction'],
        'robustness': 'calibrate uncertainty on separate simulated worlds and test on unseen prior shifts/anomalies',
    },
    'blocking_gap_for_directional_claim': 'The existing ORDEM output has altitude response only and near-duplicate azimuth shapes. Inclination/direction claims require a response library with orbital-direction information.',
}

if not has_asset_exposure:
    raise RuntimeError('No real active-satellite exposure overlapped the modeled altitude shells; inspect the propagation outputs.')

(OUTPUT_DIR / 'dynamic_model_readiness.json').write_text(json.dumps(readiness, indent=2))

print('READINESS DECISION')
print('  Real active-satellite layer loaded:', readiness['model_decisions']['use_real_active_satellites'])
print('  Use 81 independent actions:', readiness['model_decisions']['use_81_actions_as_independent_choices'])
print('  Use 9 elevation families:', readiness['model_decisions']['use_9_elevation_family_actions'])
print('  Full direction/inclination claim:', readiness['model_decisions']['claim_full_azimuth_direction_learning'])
print('  Collision-warning claim:', readiness['model_decisions']['claim_collision_warning_or_avoidance'])


## 6. Publication diagnostics and completion marker


In [ ]:
shell_centers = shell_summary.altitude_center_km.to_numpy()
fig, axes = plt.subplots(2, 2, figsize=(13.2, 9.2), constrained_layout=True)
ax = axes[0, 0]
for action_index, action in effective_actions.iterrows():
    shape = effective_H[action_index] / effective_H[action_index].sum()
    ax.plot(shell_centers, shape, marker='o', lw=1.5, label=f'EL {action.elevation_deg:g}°')
ax.set_title('A. Nine effective ORDEM elevation-response shapes')
ax.set_xlabel('Orbital altitude shell center (km)')
ax.set_ylabel('Normalized response')
ax.grid(alpha=0.2)
ax.legend(frameon=False, ncol=3, fontsize=8)
ax = axes[0, 1]
singular_fraction = shape_energy / shape_energy.sum()
ax.bar(np.arange(1, len(singular_fraction) + 1), 100 * singular_fraction, color='#376795')
ax.plot(np.arange(1, len(singular_fraction) + 1), 100 * np.cumsum(singular_fraction), color='#d1495b', marker='o', label='Cumulative')
ax.axhline(99, color='black', linestyle='--', lw=1)
ax.set_title('B. Independent information in the 81 labeled actions')
ax.set_xlabel('Singular component')
ax.set_ylabel('Response-shape energy (%)')
ax.set_xlim(0.4, min(15.5, len(singular_fraction) + 0.5))
ax.grid(axis='y', alpha=0.2)
ax.legend(frameon=False)
ax = axes[1, 0]
ax.bar(shell_centers, shell_summary.unique_active_satellites, width=12, color='#2a9d8f')
ax.set_title('C. Real active satellites intersecting each shell')
ax.set_xlabel('Orbital altitude shell center (km)')
ax.set_ylabel('Unique active NORAD IDs')
ax.grid(axis='y', alpha=0.2)
ax = axes[1, 1]
ax.plot(shell_centers, 100 * shell_summary.ordem_population_weight, marker='o', lw=2, label='ORDEM debris shape')
ax.plot(shell_centers, 100 * shell_summary.active_asset_exposure_weight, marker='s', lw=2, label='Real active-satellite exposure')
ax.set_title('D. Debris population versus protected-asset exposure')
ax.set_xlabel('Orbital altitude shell center (km)')
ax.set_ylabel('Share inside modeled window (%)')
ax.grid(alpha=0.2)
ax.legend(frameon=False)
figure_png = OUTPUT_DIR / 'dynamic_asset_action_audit_2x2.png'
figure_pdf = OUTPUT_DIR / 'dynamic_asset_action_audit_2x2.pdf'
fig.savefig(figure_png, dpi=220, bbox_inches='tight')
fig.savefig(figure_pdf, bbox_inches='tight')
plt.show()
readme = f'research dynamic active-satellite exposure and action audit\n========================================================\n\nHeadline results\n----------------\n- {len(active_omm):,} current active-satellite OMM rows loaded from a cached CelesTrak snapshot.\n- {len(orbit_summary):,} active satellites propagated over {PROPAGATION_HOURS} hours.\n- {satellites_in_window:,} active satellites entered the modeled 915–1065 km orbital-altitude window.\n- 81 labeled ORDEM actions reduce to {len(effective_actions)} meaningful elevation families.\n- Median same-elevation/different-azimuth response similarity: {np.median(same_elevation_pairs):.6f}.\n- Full azimuth/inclination debris-direction learning is not supported by this response library.\n\nCorrect interpretation\n----------------------\nThe hidden-debris state describes orbital population shells, not fixed geographic cells.\nThe active-satellite layer uses real OMM orbital elements, not invented satellite locations.\nLater hidden worlds are simulated because the subcatalog debris truth is unobserved.\nThis work supports population-risk mapping and sensor scheduling, not individual conjunction alerts.\n'
(OUTPUT_DIR / 'README_dynamic_asset_action_audit.txt').write_text(readme)
quality_checks = {'ordem_actions_equal_81': bool(response.action_id.nunique() == 81), 'effective_elevation_families_equal_9': bool(len(effective_actions) == 9), 'same_elevation_azimuth_similarity_exceeds_0p99': bool(np.median(same_elevation_pairs) > 0.99), 'active_catalog_has_required_rows': bool(len(active_omm) >= MIN_ACTIVE_ROWS), 'at_least_90_percent_propagated': bool(len(orbit_summary) >= 0.9 * len(active_omm)), 'active_satellites_overlap_modeled_window': bool(satellites_in_window > 0), 'finite_shell_exposure': bool(np.isfinite(shell_summary.select_dtypes(include=[np.number]).to_numpy()).all()), 'output_figure_exists': bool(figure_png.is_file() and figure_pdf.is_file())}
failed = [name for name, passed in quality_checks.items() if not passed]
if failed:
    raise RuntimeError(f'Final quality checks failed: {failed}')
success = {'status': 'complete', 'created_utc': datetime.now(timezone.utc).isoformat(), 'notebook': 'DEBRIS_Dynamic_Active_Satellite_Exposure_and_Action_Audit.ipynb', 'quality_checks': quality_checks, 'headline': {'active_satellite_rows': int(len(active_omm)), 'active_satellites_propagated': int(len(orbit_summary)), 'active_satellites_intersecting_window': satellites_in_window, 'raw_ordem_actions': 81, 'effective_elevation_actions': int(len(effective_actions)), 'same_elevation_azimuth_similarity_median': float(np.median(same_elevation_pairs))}}
(OUTPUT_DIR / '_SUCCESS.json').write_text(json.dumps(success, indent=2))
print('SUCCESS — dynamic asset/action audit is complete.')
print('Main outputs:')
for name in ['dynamic_model_readiness.json', 'active_satellite_shell_exposure.csv', 'active_satellite_altitude_inclination_exposure.csv', 'ordem_effective_elevation_actions.csv', 'dynamic_asset_action_audit_2x2.png', '_SUCCESS.json']:
    print(' ', OUTPUT_DIR / name)


## Outputs
